# Multi-Regulation RAG — Phase 6b: Evaluation Framework

Extends Phase 6 (DORA-only) to evaluate all three regulations:
- **DORA** — 30 questions (existing, unchanged)
- **NIS2** — 15 questions across 5 types
- **PS21/3** — 15 questions across 5 types
- **Cross-regulation** — 10 comparison questions

**Key changes from Phase 6:**
- Generation LLM: Gemini Flash (replaces Groq — no token limit)
- Judge LLM: Gemini Flash direct prompt (replaces LlamaIndex evaluators)
- Answer cache: separate files per regulation
- Scorecard: combined multi-regulation report

**Prerequisites:** Phases 1–5b complete.
Expects: `dora_rag/pipeline_config.json`, `dora_rag/dora_chunks/all_chunks.json`

In [ ]:
# API NOTE
# The Gemini API key configured below is a paid-tier key with a fixed budget cap.
# It is intentionally set to a spending limit sufficient to run this pipeline.
# By running this code, you are authorised to make API calls under that budget.
# Do not share this key publicly.

In [25]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'llama-index',
    'llama-index-llms-groq',
    'llama-index-llms-google-genai',
    'llama-index-embeddings-huggingface',
    'llama-index-vector-stores-chroma',
    'llama-index-retrievers-bm25',
    'chromadb', 'sentence-transformers', 'torch',
    'rank-bm25', 'PyStemmer', 'google-generativeai',
])
print('Dependencies installed.')


Dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [27]:
import os
# Cell 2 — Imports, config, LLM setup
#
# Generation + Judge LLM : Gemini Flash (1M context, no token limits)
# Router LLM             : Groq 8b-instant (prevents LlamaIndex OpenAI fallback)

import json, time, datetime, re, torch
from pathlib import Path
from collections import Counter, defaultdict
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.llms.google_genai import GoogleGenAI

BASE_DIR    = Path('./dora_rag')
CHUNKS_DIR  = BASE_DIR / 'dora_chunks'
CHUNKS_FILE = CHUNKS_DIR / 'all_chunks.json'   # all 3 regulations
EVAL_DIR    = BASE_DIR / 'eval'
EVAL_DIR.mkdir(exist_ok=True)

GROQ_API_KEY   = os.environ.get('GROQ_API_KEY', '')  # set in your environment or a .env file
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')  # set in your environment or a .env file
GEMINI_MODEL   = 'models/gemini-flash-latest'  
RETRIEVAL_TOP_K = 8
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Router: Groq 8b (prevents OpenAI fallback — never used for generation)
router_llm   = Groq(model='llama-3.1-8b-instant', api_key=GROQ_API_KEY)
Settings.llm = router_llm
print('✓ Settings.llm: Groq 8b (router only)')

# Generation + Judge: Gemini Flash
llm       = GoogleGenAI(model=GEMINI_MODEL, api_key=GEMINI_API_KEY)
judge_llm = llm   # same model for judging — no need for separate lightweight model
print(f'✓ Generation + Judge LLM: {GEMINI_MODEL}')

index_config      = json.loads((BASE_DIR / 'index_config.json').read_text())
CHROMA_DIR        = Path(index_config['chroma_dir'])
CHROMA_COLLECTION = index_config['chroma_collection']
EMBED_MODEL_NAME  = index_config['embed_model']

print(f'✓ Index  : {CHROMA_COLLECTION} ({index_config["total_vectors"]} vectors)')
print(f'✓ Corpus : {index_config.get("corpus_regulations", {})}')
print(f'✓ Device : {device}')
print(f'✓ Eval output: {EVAL_DIR}')


✓ Settings.llm: Groq 8b (router only)
✓ Generation + Judge LLM: models/gemini-flash-latest
✓ Index  : dora_regulation (798 vectors)
✓ Corpus : {'dora': 381, 'nis2': 179, 'ps21_3': 119, 'cross_reg': 119}
✓ Device : cpu
✓ Eval output: dora_rag/eval


In [30]:
# Cell 3 — Load embedding model, index, and multi-regulation retrieval pipeline
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.schema import TextNode
from llama_index.core.retrievers import VectorIndexRetriever, QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator
import chromadb

print('Loading embedding model...')
embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_NAME,
    query_instruction='Represent this sentence for searching relevant passages: ',
    embed_batch_size=16, device=device,
)
Settings.embed_model = embed_model

chroma_client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
chroma_collection = chroma_client.get_collection(CHROMA_COLLECTION)
vector_store      = ChromaVectorStore(chroma_collection=chroma_collection)
index             = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=embed_model)

raw_chunks = json.loads(CHUNKS_FILE.read_text(encoding='utf-8'))
all_nodes  = [
    TextNode(
        id_=c['chunk_id'], text=c['text_with_header'],
        metadata={
            'regulation':    c.get('regulation', 'dora'),
            'jurisdiction':  c.get('jurisdiction', 'eu'),
            'source_doc':    c['source_doc'],
            'doc_type':      c['doc_type'],
            'level':         c.get('level', 0),
            'article_num':   c['article_num'] if c['article_num'] is not None else -1,
            'article_title': c['article_title'],
            'section_ref':   c.get('section_ref') or '',
            'paragraph_idx': c['paragraph_idx'],
            'topic_tags':    '|'.join(c['topic_tags']),
            'entity_scope':  '|'.join(c['entity_scope']),
            'token_estimate':c['token_estimate'],
        }
    ) for c in raw_chunks
]

try:
    import Stemmer
    stemmer = Stemmer.Stemmer('english')
except ImportError:
    stemmer = None

def make_retriever(level=None, doc_type=None, regulation=None, jurisdiction=None, top_k=8):
    filters = []
    if level        is not None: filters.append(MetadataFilter(key='level',        value=level,        operator=FilterOperator.EQ))
    if doc_type     is not None: filters.append(MetadataFilter(key='doc_type',     value=doc_type,     operator=FilterOperator.EQ))
    if regulation   is not None: filters.append(MetadataFilter(key='regulation',   value=regulation,   operator=FilterOperator.EQ))
    if jurisdiction is not None: filters.append(MetadataFilter(key='jurisdiction', value=jurisdiction, operator=FilterOperator.EQ))
    if filters:
        return VectorIndexRetriever(index=index, similarity_top_k=top_k,
                                    filters=MetadataFilters(filters=filters))
    sem = VectorIndexRetriever(index=index, similarity_top_k=top_k)
    bm  = BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k,
                                       stemmer=stemmer, language='english')
    return QueryFusionRetriever(
        retrievers=[sem, bm], llm=Settings.llm, similarity_top_k=top_k,
        num_queries=1, mode='reciprocal_rerank', use_async=False, verbose=False,
    )

def detect_regulation(query):
    q, d = query.lower(), []
    if any(k in q for k in ['dora','2022/2554','digital operational resilience','financial entit']): d.append('dora')
    if any(k in q for k in ['nis2','nis 2','2022/2555','essential entit','important entit','enisa']): d.append('nis2')
    if any(k in q for k in ['ps21','ps21/3','ss1/21','fg22','fca','pra','impact tolerance','important business service']): d.append('ps21_3')
    return d

def route_query(query, regulation=None):
    q = query.lower()
    if regulation:
        reg_scope = regulation
    else:
        detected  = detect_regulation(query)
        reg_scope = detected[0] if len(detected) == 1 else None
    if reg_scope == 'nis2':
        if any(k in q for k in ['implementing regulation','technical requirement','2024/2690']): return make_retriever(regulation='nis2', level=2, top_k=10), 'NIS2 L2'
        if any(k in q for k in ['enisa','guideline','security measure']): return make_retriever(regulation='nis2', level=3, top_k=8), 'NIS2 L3'
        return make_retriever(regulation='nis2', top_k=10), 'NIS2 full'
    if reg_scope == 'ps21_3':
        if any(k in q for k in ['ss1/21','supervisory statement']): return make_retriever(regulation='ps21_3', doc_type='supervisory_statement', top_k=8), 'PS21/3 SS'
        if any(k in q for k in ['fg22','finalised guidance']): return make_retriever(regulation='ps21_3', doc_type='guidance', top_k=8), 'PS21/3 FG'
        return make_retriever(regulation='ps21_3', top_k=10), 'PS21/3 full'
    reg_filter = 'dora' if reg_scope == 'dora' else None
    if any(k in q for k in ['regulatory technical standard',' rts ','rts on','technical standard','delegated regulation']): return make_retriever(level=2, regulation=reg_filter, top_k=12), 'L2 (RTS/ITS)'
    if any(k in q for k in ['tlpt','penetration test','threat-led','tiber']): return make_retriever(regulation=reg_filter, top_k=12), 'Full (TLPT)'
    if any(k in q for k in ['subcontract','chain','group entit','third-party provider']): return make_retriever(regulation=reg_filter, top_k=12), 'Full (adversarial)'
    if any(k in q for k in ['guideline','aggregate','oversight cooperation']): return make_retriever(level=3, regulation=reg_filter, top_k=8), 'L3'
    if any(k in q for k in ['template','reporting form']): return make_retriever(doc_type='ITS', regulation=reg_filter, top_k=8), 'ITS'
    if any(k in q for k in ['article ','regulation ','directive ',' shall ',' must ']): return make_retriever(level=1, regulation=reg_filter, top_k=8), 'L1'
    if reg_filter: return make_retriever(regulation=reg_filter, top_k=8), f'{reg_filter.upper()} full'
    return make_retriever(top_k=10), 'All regulations'

def retrieve(query, regulation=None):
    retriever, scope = route_query(query, regulation=regulation)
    return retriever.retrieve(query), scope

print(f'✓ Pipeline loaded — {chroma_collection.count()} vectors, {len(all_nodes)} nodes')
by_reg = Counter(c.get('regulation','?') for c in raw_chunks)
for reg, cnt in sorted(by_reg.items()):
    print(f'  {reg:<12}: {cnt} chunks')


Loading embedding model...
✓ Pipeline loaded — 798 vectors, 826 nodes
  cross_reg   : 128 chunks
  dora        : 390 chunks
  nis2        : 189 chunks
  ps21_3      : 119 chunks


In [31]:
# Cell 4 — Multi-regulation prompt and ask() function
from llama_index.core import PromptTemplate

# DORA prompt (unchanged from Phase 6 — used for DORA eval questions)
DORA_QA_PROMPT = PromptTemplate("""\
You are a regulatory compliance assistant specialising in EU DORA \
(Digital Operational Resilience Act - Regulation (EU) 2022/2554) \
and its Level 2 regulatory technical standards and Level 3 guidelines.

CONTEXT (retrieved regulatory provisions):
---------------------------------------------
{context_str}
---------------------------------------------

QUESTION: {query_str}

INSTRUCTIONS - follow ALL rules exactly, no exceptions:
1. Base your answer ONLY on the context provided above. Never use outside knowledge.
2. Cite every statement: [Source: <document_name>, Article <N>]
3. Distinguish: SHALL/MUST = mandatory, SHOULD = recommended, MAY = optional.
4. Flag entity-specific applicability where relevant.
5. STRICT REFUSAL RULE: If the primary topic is NOT addressed in the context,
   respond with ONLY: 'INSUFFICIENT CONTEXT: The DORA regulatory corpus does not
   directly address this question. This topic falls outside the scope of DORA.'
6. Structure: direct answer first, then detail with citations.

ANSWER:\
""")

# Multi-regulation prompt — used for NIS2, PS21/3 eval questions
MULTI_REG_PROMPT = PromptTemplate("""\
You are a regulatory compliance expert specialising in {regulation_label}.

CONTEXT (retrieved regulatory provisions):
---------------------------------------------
{context_str}
---------------------------------------------

QUESTION: {query_str}

INSTRUCTIONS:
1. Answer ONLY from the context above. Never use outside knowledge.
2. Cite every statement: [Source: <document>, Article <N>] for EU docs,
   [Source: <document>, Section <ref>] for UK PS21/3 docs.
3. Distinguish: SHALL/MUST = mandatory, SHOULD = recommended, MAY = optional.
4. STRICT REFUSAL: If the topic falls outside {regulation_label} scope,
   respond ONLY with: 'INSUFFICIENT CONTEXT: This topic falls outside the scope
   of {regulation_label}.'
5. Structure: direct answer first, then detail with citations.

ANSWER:\
""")

REG_LABELS = {
    'dora':   'EU DORA (Regulation 2022/2554)',
    'nis2':   'EU NIS2 (Directive 2022/2555)',
    'ps21_3': 'UK PS21/3 (FCA/PRA Operational Resilience)',
}


def llm_complete_with_retry(prompt, max_retries=4):
    for attempt in range(1, max_retries + 1):
        try:
            return llm.complete(prompt).text.strip()
        except Exception as e:
            err = str(e)
            is_rate = any(k in err for k in ['RESOURCE_EXHAUSTED','429','quota','rate_limit','Too Many Requests'])
            if not is_rate or attempt == max_retries: raise
            match = re.search(r'retry_delay.*?(\d+)', err)
            wait  = int(match.group(1)) + 2 if match else 30
            print(f'    Rate limit ({attempt}/{max_retries}). Waiting {wait}s...')
            time.sleep(wait)


def _build_context(nodes):
    parts = []
    for node in nodes:
        m   = node.metadata
        src = m.get('source_doc', 'Unknown')
        art = m.get('article_num', -1)
        sec = m.get('section_ref', '')
        ttl = m.get('article_title', '')
        if art != -1:  hdr = f'[{src}, Article {art} — {ttl}]'
        elif sec:      hdr = f'[{src}, Section {sec} — {ttl}]'
        else:          hdr = f'[{src}]'
        parts.append(f'{hdr}\n{node.text}')
    return '\n\n'.join(parts)


def ask(question, regulation=None, verbose=False):
    """Single-regulation ask. regulation=None defaults to DORA (Phase 6 compatible)."""
    t0 = time.time()
    nodes, scope = retrieve(question, regulation=regulation)
    context_str  = _build_context(nodes)

    if regulation and regulation != 'dora':
        prompt = MULTI_REG_PROMPT.format(
            regulation_label=REG_LABELS.get(regulation, regulation),
            context_str=context_str, query_str=question
        )
    else:
        prompt = DORA_QA_PROMPT.format(context_str=context_str, query_str=question)

    answer  = llm_complete_with_retry(prompt)
    latency = round(time.time() - t0, 2)
    result  = {'question': question, 'answer': answer, 'nodes': nodes,
               'scope': scope, 'latency_s': latency}
    if verbose:
        print(f"\n{'='*65}\nQ: {question}\nScope: {scope} | {latency}s")
        print('-' * 65)
        print(answer)
        print('=' * 65)
    return result


print('✓ Multi-regulation ask() ready (Gemini generation)')
print('  ask(question)                    — DORA (Phase 6 compatible)')
print('  ask(question, regulation="nis2") — NIS2')
print('  ask(question, regulation="ps21_3") — PS21/3')


✓ Multi-regulation ask() ready (Gemini generation)
  ask(question)                    — DORA (Phase 6 compatible)
  ask(question, regulation="nis2") — NIS2
  ask(question, regulation="ps21_3") — PS21/3


In [32]:
# Cell 5 — Multi-regulation evaluation question set
#
# Format: (question, query_type, regulation, expected_source_contains, expected_article, is_negative)
#
# DORA_QUESTIONS   : 30 questions (unchanged from Phase 6)
# NIS2_QUESTIONS   : 15 questions across 5 types
# PS213_QUESTIONS  : 15 questions across 5 types
# CROSS_QUESTIONS  : 10 cross-regulation questions

# ── DORA (unchanged from Phase 6) ────────────────────────────────────────────
DORA_QUESTIONS = [
    ('What are the reporting timelines for a major ICT-related incident?',                          'factual',        'dora', 'L1_DORA_Regulation', 19, False),
    ('What information must be included in the initial notification of a major ICT incident?',      'factual',        'dora', 'L1_DORA_Regulation', 19, False),
    ('What is the minimum content of an ICT business continuity policy under DORA?',                'factual',        'dora', 'L1_DORA_Regulation', 11, False),
    ('What must the ICT risk management framework include at a minimum?',                           'factual',        'dora', 'L1_DORA_Regulation',  6, False),
    ('What are the general principles for ICT third-party risk management?',                        'factual',        'dora', 'L1_DORA_Regulation', 28, False),
    ('Is threat-led penetration testing mandatory for all financial entities?',                     'obligation',     'dora', 'L1_DORA_Regulation', 26, False),
    ('Must financial entities always notify customers about major ICT incidents?',                  'obligation',     'dora', 'L1_DORA_Regulation', None, False),
    ('Is it mandatory to include exit strategies in ICT third-party contracts?',                    'obligation',     'dora', 'L1_DORA_Regulation', 30, False),
    ('Are financial entities required to test their ICT business continuity plans?',                'obligation',     'dora', 'L1_DORA_Regulation', 11, False),
    ('Can financial entities voluntarily share cyber threat intelligence?',                         'obligation',     'dora', 'L1_DORA_Regulation', 45, False),
    ('Does DORA apply to small non-interconnected investment firms?',                               'applicability',  'dora', 'L1_DORA_Regulation', None, False),
    ('What DORA obligations apply to crypto-asset service providers?',                              'applicability',  'dora', 'L1_DORA_Regulation', None, False),
    ('Which entities are subject to the simplified ICT risk management framework?',                 'applicability',  'dora', 'L1_DORA_Regulation', None, False),
    ('Are ICT third-party providers directly subject to DORA?',                                    'applicability',  'dora', 'L1_DORA_Regulation', None, False),
    ('Which financial entities must conduct TLPT?',                                                 'applicability',  'dora', 'L1_DORA_Regulation', 26, False),
    ('How does the RTS on ICT incident classification specify the criteria in the main regulation?','cross_level',    'dora', 'L2_RTS_ICT_Incident_Classification', None, False),
    ('What additional detail does the RTS on subcontracting provide beyond Article 28?',            'cross_level',    'dora', 'L2_RTS_Subcontracting', None, False),
    ('How does the TLPT RTS expand on Article 26 of the main regulation?',                          'cross_level',    'dora', 'L2_RTS_TLPT', None, False),
    ('What does the RTS on ICT risk management say about governance requirements?',                 'cross_level',    'dora', 'L2_RTS_ICT_Risk_Management', None, False),
    ('What contractual provisions must be included in ICT third-party contracts under Article 30?', 'cross_reference','dora', 'L1_DORA_Regulation', 30, False),
    ('How do Articles 19 and 20 relate to each other in the incident reporting process?',           'cross_reference','dora', 'L1_DORA_Regulation', 19, False),
    ('What is the relationship between the ICT risk management framework and business continuity?', 'cross_reference','dora', 'L1_DORA_Regulation', None, False),
    ('What does DORA say about GDPR data subject access requests?',                                 'negative',       'dora', None, None, True),
    ('What are the DORA requirements for ESG reporting?',                                           'negative',       'dora', None, None, True),
    ('How does DORA interact with the EU AI Act?',                                                  'negative',       'dora', None, None, True),
    ('A cloud provider subcontracts storage to a third party. What DORA obligations apply?',        'adversarial',    'dora', 'L1_DORA_Regulation', None, False),
    ('Can a single TLPT satisfy requirements for multiple group entities?',                         'adversarial',    'dora', 'L2_RTS_TLPT', None, False),
    ('What happens if a financial entity misclassifies a major incident as minor?',                 'adversarial',    'dora', 'L1_DORA_Regulation', None, False),
    ('How should ICT risk tolerance be documented to satisfy DORA requirements?',                   'adversarial',    'dora', 'L1_DORA_Regulation', None, False),
    ('What are the consequences of failing to submit the final incident report on time?',           'adversarial',    'dora', 'L1_DORA_Regulation', None, False),
]

# ── NIS2 (15 questions) ───────────────────────────────────────────────────────
NIS2_QUESTIONS = [
    # Factual
    ('What are the NIS2 incident reporting timelines under Article 23?',                            'factual',        'nis2', 'NIS2_Directive', 23, False),
    ('What cybersecurity risk management measures must entities implement under Article 21?',       'factual',        'nis2', 'NIS2_Directive', 21, False),
    ('What are the technical requirements under Commission IR 2024/2690?',                         'factual',        'nis2', 'NIS2_IR',       None, False),
    # Obligation
    ('Is multi-factor authentication mandatory under NIS2?',                                        'obligation',     'nis2', 'NIS2_Directive', 21, False),
    ('Must essential entities conduct regular security assessments under NIS2?',                    'obligation',     'nis2', 'NIS2_Directive', 21, False),
    # Applicability
    ('Which entities qualify as essential entities under NIS2?',                                    'applicability',  'nis2', 'NIS2_Directive',  3, False),
    ('Does NIS2 apply to DNS service providers?',                                                   'applicability',  'nis2', 'NIS2_Directive', None, False),
    ('What is the difference between essential and important entities under NIS2?',                 'applicability',  'nis2', 'NIS2_Directive', None, False),
    # Cross-level
    ('How does IR 2024/2690 specify the security measures required by Article 21?',                 'cross_level',    'nis2', 'NIS2_IR',       None, False),
    ('What supply chain security obligations does NIS2 impose?',                                    'cross_level',    'nis2', 'NIS2_Directive', 21, False),
    # Adversarial
    ('A cloud provider operates in multiple EU member states. Which NIS2 obligations apply?',       'adversarial',    'nis2', 'NIS2_Directive', None, False),
    ('What are the NIS2 sanctions for non-compliance by essential entities?',                       'adversarial',    'nis2', 'NIS2_Directive', None, False),
    # Negative
    ('What does NIS2 say about impact tolerances for important business services?',                 'negative',       'nis2', None, None, True),
    ('What are the NIS2 requirements for TLPT threat-led penetration testing?',                     'negative',       'nis2', None, None, True),
    ('What does NIS2 say about the ICT register of information?',                                   'negative',       'nis2', None, None, True),
]

# ── PS21/3 (15 questions) ─────────────────────────────────────────────────────
PS213_QUESTIONS = [
    # Factual
    ('What is an impact tolerance and how should firms set one under PS21/3?',                      'factual',        'ps21_3', 'PS21_3', None, False),
    ('What does SS1/21 require firms to include in their self-assessment?',                         'factual',        'ps21_3', 'SS1_21', None, False),
    ('What are the PS21/3 requirements for mapping important business services?',                   'factual',        'ps21_3', 'PS21_3', None, False),
    # Obligation
    ('Is scenario testing mandatory for all FCA/PRA regulated firms?',                              'obligation',     'ps21_3', 'PS21_3', None, False),
    ('Must firms submit their operational resilience self-assessment to the regulator?',            'obligation',     'ps21_3', 'SS1_21', None, False),
    # Applicability
    ('Which firms are subject to PS21/3 operational resilience requirements?',                      'applicability',  'ps21_3', 'PS21_3', None, False),
    ('Does PS21/3 apply to non-systemic smaller firms?',                                            'applicability',  'ps21_3', 'PS21_3', None, False),
    # Cross-level
    ('How does SS1/21 elaborate on the impact tolerance requirements in PS21/3?',                   'cross_level',    'ps21_3', 'SS1_21', None, False),
    ('What additional guidance does FG22/5 provide on operational resilience?',                     'cross_level',    'ps21_3', 'FCA_FG22', None, False),
    # Adversarial
    ('A firm cannot stay within its impact tolerance during a severe scenario. What must it do?',   'adversarial',    'ps21_3', 'SS1_21', None, False),
    ('What are the PS21/3 requirements for critical third parties?',                                'adversarial',    'ps21_3', 'DP3_22', None, False),
    # Negative
    ('What does PS21/3 say about incident reporting timelines to competent authorities?',           'negative',       'ps21_3', None, None, True),
    ('What are the PS21/3 requirements for TLPT penetration testing?',                              'negative',       'ps21_3', None, None, True),
    ('What does PS21/3 say about the DORA register of information requirement?',                    'negative',       'ps21_3', None, None, True),
    ('What are the PS21/3 sanctions for firms that breach impact tolerances?',                      'negative',       'ps21_3', None, None, True),
]

# ── Cross-regulation (10 questions) ──────────────────────────────────────────
CROSS_QUESTIONS = [
    # These are answered using ask_comparison() — all 3 regulations
    ('What are the incident reporting obligations and timelines?',                                  'comparison', None, None, None, False),
    ('What third-party and supply chain risk management is required?',                              'comparison', None, None, None, False),
    ('What governance and board-level responsibilities are required?',                              'comparison', None, None, None, False),
    ('What testing requirements apply and how do they compare?',                                    'comparison', None, None, None, False),
    ('Which types of firms are subject to each framework?',                                         'comparison', None, None, None, False),
    ('What are the cybersecurity risk management requirements?',                                    'comparison', None, None, None, False),
    ('What business continuity requirements apply under each framework?',                           'comparison', None, None, None, False),
    ('How do the three frameworks define and handle major incidents?',                              'comparison', None, None, None, False),
    ('What obligations apply to a UK bank that also operates in the EU?',                          'comparison', None, None, None, False),
    ('Where do DORA and NIS2 overlap for financial entities subject to both?',                     'comparison', None, None, None, False),
]

# Combined question set
EVAL_QUESTIONS = DORA_QUESTIONS + NIS2_QUESTIONS + PS213_QUESTIONS + CROSS_QUESTIONS

by_reg  = Counter(q[2] for q in EVAL_QUESTIONS)
by_type = Counter(q[1] for q in EVAL_QUESTIONS)
print(f'Total evaluation questions: {len(EVAL_QUESTIONS)}')
print(f'\nBy regulation:')
for reg, cnt in sorted(by_reg.items(), key=lambda x: str(x[0])):
    print(f'  {str(reg):<12}: {cnt}')
print(f'\nBy type:')
for qtype, cnt in by_type.most_common():
    print(f'  {qtype:<20}: {cnt}')


Total evaluation questions: 70

By regulation:
  None        : 10
  dora        : 30
  nis2        : 15
  ps21_3      : 15

By type:
  factual             : 11
  applicability       : 10
  negative            : 10
  comparison          : 10
  obligation          : 9
  adversarial         : 9
  cross_level         : 8
  cross_reference     : 3


In [33]:
# Cell 6 — Retrieval evaluation (no LLM calls)
# Evaluates DORA, NIS2, and PS21/3 question sets separately.
# Cross-regulation questions are skipped (they use ask_comparison, not single retrieval).

def eval_retrieval(questions, label):
    results = []
    for q_tuple in questions:
        question, qtype, regulation, exp_src, exp_art, is_neg = q_tuple
        if is_neg:
            continue
        nodes, scope = retrieve(question, regulation=regulation)
        retrieved_srcs = [n.metadata.get('source_doc','') for n in nodes]
        retrieved_arts = [n.metadata.get('article_num',-1) for n in nodes]
        src_hit = exp_src is None or any(exp_src in s for s in retrieved_srcs)
        art_hit = exp_art is None or exp_art in retrieved_arts
        # Reciprocal Rank
        rr = 0.0
        if exp_src:
            for rank, s in enumerate(retrieved_srcs, 1):
                if exp_src in s:
                    rr = 1.0 / rank
                    break
        # Precision@5
        top5_srcs = retrieved_srcs[:5]
        p5 = sum(1 for s in top5_srcs if exp_src and exp_src in s) / 5 if exp_src else 0
        results.append({'question': question, 'type': qtype, 'regulation': regulation,
                         'src_hit': src_hit, 'art_hit': art_hit, 'rr': rr, 'p5': p5})
    if not results:
        return {}
    src_rate = sum(1 for r in results if r['src_hit']) / len(results)
    art_rate = sum(1 for r in results if r['art_hit']) / len(results)
    mrr      = sum(r['rr'] for r in results) / len(results)
    avg_p5   = sum(r['p5'] for r in results) / len(results)
    print(f'  {label:<12}: n={len(results)}  Source={src_rate:.0%}  Article={art_rate:.0%}  MRR={mrr:.3f}  P@5={avg_p5:.2f}')
    return {'results': results, 'src_rate': src_rate, 'art_rate': art_rate, 'mrr': mrr, 'avg_p5': avg_p5}


print('Running retrieval evaluation (no LLM calls)...')
print(f"  {'Corpus':<12}  {'n':>3}  {'Source':>8}  {'Article':>8}  {'MRR':>6}  {'P@5':>5}")
print('  ' + '─' * 55)

dora_ret  = eval_retrieval(DORA_QUESTIONS,  'DORA')
nis2_ret  = eval_retrieval(NIS2_QUESTIONS,  'NIS2')
ps213_ret = eval_retrieval(PS213_QUESTIONS, 'PS21/3')

print()
print('RETRIEVAL SUMMARY')
print('=' * 55)
for label, ret in [('DORA', dora_ret), ('NIS2', nis2_ret), ('PS21/3', ps213_ret)]:
    if ret:
        print(f"  {label:<8} Source={ret['src_rate']:.0%}  Article={ret['art_rate']:.0%}  MRR={ret['mrr']:.3f}")
        status_src = '✓' if ret['src_rate'] >= 0.8 else '✗'
        status_art = '✓' if ret['art_rate'] >= 0.7 else '✗'
        print(f'           Source ≥80%: {status_src}   Article ≥70%: {status_art}')
print()
# Store combined for use in scorecard
retrieval_results_all = {
    'dora': dora_ret, 'nis2': nis2_ret, 'ps21_3': ps213_ret
}


Running retrieval evaluation (no LLM calls)...
  Corpus          n    Source   Article     MRR    P@5
  ───────────────────────────────────────────────────────
  DORA        : n=27  Source=93%  Article=89%  MRR=0.590  P@5=0.48
  NIS2        : n=12  Source=92%  Article=83%  MRR=0.861  P@5=0.72
  PS21/3      : n=11  Source=100%  Article=100%  MRR=0.874  P@5=0.71

RETRIEVAL SUMMARY
  DORA     Source=93%  Article=89%  MRR=0.590
           Source ≥80%: ✓   Article ≥70%: ✓
  NIS2     Source=92%  Article=83%  MRR=0.861
           Source ≥80%: ✓   Article ≥70%: ✓
  PS21/3   Source=100%  Article=100%  MRR=0.874
           Source ≥80%: ✓   Article ≥70%: ✓



In [34]:
# Cell 7 — Retrieval breakdown by query type (per regulation)

print('RETRIEVAL BREAKDOWN BY QUERY TYPE')
print('=' * 65)

for reg, reg_questions in [('dora', DORA_QUESTIONS), ('nis2', NIS2_QUESTIONS), ('ps21_3', PS213_QUESTIONS)]:
    ret = retrieval_results_all.get(reg, {})
    results = ret.get('results', [])
    if not results:
        continue

    by_type_results = defaultdict(list)
    for r in results:
        by_type_results[r['type']].append(r)

    print(f'\n  [{reg.upper()}]')
    print(f"  {'Type':<20} {'n':>3}  {'Src':>5}  {'Art':>5}  {'MRR':>6}")
    print('  ' + '─' * 45)

    for qtype, rows in sorted(by_type_results.items()):
        src = sum(1 for r in rows if r['src_hit']) / len(rows)
        art = sum(1 for r in rows if r['art_hit']) / len(rows)
        mrr = sum(r['rr'] for r in rows) / len(rows)
        print(f"  {qtype:<20} {len(rows):>3}  {src:>4.0%}  {art:>4.0%}  {mrr:>6.3f}")

print()

# Also keep a flat retrieval_results list for any downstream cells that need it
retrieval_results = []
for reg_ret in retrieval_results_all.values():
    retrieval_results.extend(reg_ret.get('results', []))

# Summary metrics across all regulations
src_rate = sum(1 for r in retrieval_results if r['src_hit']) / len(retrieval_results) if retrieval_results else 0
art_rate = sum(1 for r in retrieval_results if r['art_hit']) / len(retrieval_results) if retrieval_results else 0
mrr      = sum(r['rr'] for r in retrieval_results) / len(retrieval_results) if retrieval_results else 0
avg_p5   = sum(r['p5'] for r in retrieval_results) / len(retrieval_results) if retrieval_results else 0

print(f'COMBINED RETRIEVAL METRICS (all regulations, n={len(retrieval_results)})')
print('=' * 65)
print(f'  Source hit rate  : {src_rate:.0%}')
print(f'  Article hit rate : {art_rate:.0%}')
print(f'  MRR              : {mrr:.3f}')
print(f'  Avg Precision@5  : {avg_p5:.2f}')

RETRIEVAL BREAKDOWN BY QUERY TYPE

  [DORA]
  Type                   n    Src    Art     MRR
  ─────────────────────────────────────────────
  adversarial            5   60%  100%   0.283
  applicability          5  100%  100%   0.750
  cross_level            4  100%  100%   0.650
  cross_reference        3  100%   67%   0.492
  factual                5  100%   60%   0.740
  obligation             5  100%  100%   0.600

  [NIS2]
  Type                   n    Src    Art     MRR
  ─────────────────────────────────────────────
  adversarial            2  100%  100%   1.000
  applicability          3  100%  100%   1.000
  cross_level            2  100%  100%   0.667
  factual                3  100%  100%   1.000
  obligation             2   50%    0%   0.500

  [PS21_3]
  Type                   n    Src    Art     MRR
  ─────────────────────────────────────────────
  adversarial            2  100%  100%   1.000
  applicability          2  100%  100%   0.556
  cross_level            2  100%

In [35]:
import nest_asyncio; nest_asyncio.apply()


In [36]:
# Cell 8a — Generate answers and cache (all regulations)
#
# Runs ask() for DORA, NIS2, and PS21/3 questions.
# Answers are cached to avoid re-running expensive LLM calls.
# Re-run Cell 8b independently to re-judge without regenerating.

SLEEP_BETWEEN     = 5
ANSWER_CACHE_FILE = EVAL_DIR / 'generated_answers_cache.json'

# Sample subset — adjust to run more/fewer questions
# For full run: eval_subset = DORA_QUESTIONS + NIS2_QUESTIONS + PS213_QUESTIONS
non_neg_dora  = [q for q in DORA_QUESTIONS  if not q[5]][:6]
neg_dora      = [q for q in DORA_QUESTIONS  if q[5]][:1]
non_neg_nis2  = [q for q in NIS2_QUESTIONS  if not q[5]][:4]
neg_nis2      = [q for q in NIS2_QUESTIONS  if q[5]][:1]
non_neg_ps213 = [q for q in PS213_QUESTIONS if not q[5]][:4]
neg_ps213     = [q for q in PS213_QUESTIONS if q[5]][:1]

eval_subset = non_neg_dora + neg_dora + non_neg_nis2 + neg_nis2 + non_neg_ps213 + neg_ps213
print(f'Generation eval subset: {len(eval_subset)} questions')
print(f'  DORA: {len(non_neg_dora)+len(neg_dora)}  NIS2: {len(non_neg_nis2)+len(neg_nis2)}  PS21/3: {len(non_neg_ps213)+len(neg_ps213)}')
print(f'  Estimated time: ~{len(eval_subset)*35//60 + 1} minutes\n')

gen_answers = []
for i, q_tuple in enumerate(eval_subset, 1):
    question, qtype, regulation, exp_src, exp_art, is_neg = q_tuple
    print(f'  [{i}/{len(eval_subset)}] [{regulation}] {question[:50]}...')

    result  = ask(question, regulation=regulation)
    answer  = result['answer']
    context = [n.text for n in result['nodes']]   # full context — all retrieved chunks

    # Pre-compute non-LLM checks
    is_refusal   = bool(re.search(
        r'INSUFFICIENT CONTEXT|outside the scope|does not contain',
        answer, re.IGNORECASE
    ))
    has_citation = bool(re.search(r'\[Source:', answer, re.IGNORECASE)) or is_refusal
    has_obligation = bool(re.search(r'\b(shall|must|should|may)\b', answer, re.IGNORECASE))
    hallucination_guard = (not is_neg) or is_refusal

    gen_answers.append({
        'question':           question,
        'type':               qtype,
        'regulation':         regulation,          # ← stored correctly
        'is_negative':        is_neg,
        'answer':             answer,
        'context':            context,             # ← full context, not truncated
        'latency_s':          result['latency_s'],
        'has_citation':       has_citation,        # ← refusal counts as valid
        'has_obligation':     has_obligation,
        'hallucination_guard':hallucination_guard,
    })

    print(f'     Guard={hallucination_guard} Cite={has_citation} Refusal={is_refusal} [{result["latency_s"]}s]')

    if i < len(eval_subset):
        time.sleep(SLEEP_BETWEEN)

ANSWER_CACHE_FILE.write_text(json.dumps(gen_answers, indent=2))
print(f'\n✓ {len(gen_answers)} answers cached: {ANSWER_CACHE_FILE}')
print('  Re-run Cell 8b anytime to re-judge without regenerating answers')

Generation eval subset: 17 questions
  DORA: 7  NIS2: 5  PS21/3: 5
  Estimated time: ~10 minutes

  [1/17] [dora] What are the reporting timelines for a major ICT-r...
     Guard=True Cite=True Refusal=True [5.43s]
  [2/17] [dora] What information must be included in the initial n...
     Guard=True Cite=True Refusal=False [10.75s]
  [3/17] [dora] What is the minimum content of an ICT business con...
     Guard=True Cite=True Refusal=False [16.4s]
  [4/17] [dora] What must the ICT risk management framework includ...


Retrying llama_index.llms.google_genai.base.GoogleGenAI._chat in 1 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.


     Guard=True Cite=True Refusal=False [19.18s]
  [5/17] [dora] What are the general principles for ICT third-part...
     Guard=True Cite=True Refusal=False [11.97s]
  [6/17] [dora] Is threat-led penetration testing mandatory for al...
     Guard=True Cite=True Refusal=False [7.3s]
  [7/17] [dora] What does DORA say about GDPR data subject access ...
     Guard=True Cite=True Refusal=True [7.59s]
  [8/17] [nis2] What are the NIS2 incident reporting timelines und...
     Guard=True Cite=True Refusal=False [261.74s]
  [9/17] [nis2] What cybersecurity risk management measures must e...
     Guard=True Cite=True Refusal=False [14.34s]
  [10/17] [nis2] What are the technical requirements under Commissi...
     Guard=True Cite=True Refusal=False [22.25s]
  [11/17] [nis2] Is multi-factor authentication mandatory under NIS...
     Guard=True Cite=True Refusal=False [8.88s]
  [12/17] [nis2] What does NIS2 say about impact tolerances for imp...
     Guard=False Cite=True Refusal=False [15.72s]

In [37]:
# Cell 8b — Judge answers (direct Gemini judge — no LlamaIndex evaluators)
#
# Uses a single Gemini call per question to score faithfulness and relevancy.
# Avoids the LlamaIndex FaithfulnessEvaluator async/event-loop conflicts.
# Can be re-run independently without re-running Cell 8a.

import json as _json

# Define cache path (also defined in Cell 8a — repeated here for standalone re-runs)
ANSWER_CACHE_FILE = EVAL_DIR / 'generated_answers_cache.json'
SLEEP_JUDGE = 5

if 'gen_answers' not in dir() or not gen_answers:
    if ANSWER_CACHE_FILE.exists():
        gen_answers = _json.loads(ANSWER_CACHE_FILE.read_text())
        print(f'✓ Loaded {len(gen_answers)} answers from cache')
    else:
        raise RuntimeError('Run Cell 8a first to generate and cache answers.')

JUDGE_PROMPT = """\
You are evaluating a RAG system answer for a regulatory compliance assistant.

QUESTION: {question}
REGULATION: {regulation}

RETRIEVED CONTEXT:
{context}

ANSWER:
{answer}

Score on two dimensions. Respond with ONLY this JSON (no markdown, no explanation):
{{"faithfulness": 1 or 0, "relevancy": 1 or 0, "reason": "one sentence"}}

faithfulness = 1 if the answer's claims are grounded in the context.
Score 0 ONLY if the answer states specific facts that directly contradict
the context or are completely absent from ALL provided context chunks.
Cross-references to other articles mentioned within the context are acceptable.
If the context is insufficient to verify a claim, default to faithfulness=1.
A refusal response (INSUFFICIENT CONTEXT) always scores faithfulness=1.
relevancy    = 1 if the answer directly addresses the question, else 0
Note: For negative tests, a refusal response (INSUFFICIENT CONTEXT) is correct — score faithfulness=1, relevancy=1
"""

SLEEP_JUDGE = 5
print(f'Judging {len(gen_answers)} answers with {GEMINI_MODEL}...')
print(f'1 Gemini call per question | {SLEEP_JUDGE}s between calls\n')

gen_results = []
for i, r in enumerate(gen_answers, 1):
    question   = r['question']
    answer     = r['answer']
    contexts   = r.get('context', [])
    regulation = r.get('regulation', 'dora')
    reg_label  = REG_LABELS.get(regulation, regulation)

    print(f'  [{i}/{len(gen_answers)}] [{regulation}] {question[:50]}...')

    ctx_str = '\n\n'.join(contexts)   # full context   # cap context for judge
    judge_prompt = JUDGE_PROMPT.format(
        question=question, regulation=reg_label,
        context=ctx_str, answer=answer[:2000]
    )
    try:
        judge_raw = llm_complete_with_retry(judge_prompt)
        clean     = re.sub(r'```json|```', '', judge_raw).strip()
        scores    = _json.loads(clean)
        faith_pass  = bool(scores.get('faithfulness', 0))
        rel_pass    = bool(scores.get('relevancy', 0))
        reason      = scores.get('reason', '')
        faith_score = 1.0 if faith_pass else 0.0
        rel_score   = 1.0 if rel_pass   else 0.0
    except Exception as e:
        print(f'    Judge error: {e}')
        faith_pass, rel_pass = None, None
        faith_score, rel_score, reason = 0.0, 0.0, f'error: {str(e)[:50]}'

    gen_results.append({
        **{k: v for k, v in r.items() if k != 'context'},
        'faithful':    faith_pass,
        'faith_score': faith_score,
        'relevant':    rel_pass,
        'rel_score':   rel_score,
        'reason':      reason,
    })
    f  = '✓' if faith_pass else '✗'
    rv = '✓' if rel_pass   else '✗'
    h  = '✓' if r['hallucination_guard'] else '✗'
    print(f'    Faith:{f} Rel:{rv} Guard:{h}  {reason[:60]}')

    if i < len(gen_answers):
        time.sleep(SLEEP_JUDGE)

print(f'\n✓ Judging complete ({len(gen_results)} questions)')


Judging 17 answers with models/gemini-flash-latest...
1 Gemini call per question | 5s between calls

  [1/17] [dora] What are the reporting timelines for a major ICT-r...
    Faith:✓ Rel:✓ Guard:✓  The provided context describes the data fields and templates
  [2/17] [dora] What information must be included in the initial n...
    Faith:✓ Rel:✓ Guard:✓  The answer accurately reflects the requirements in Article 1
  [3/17] [dora] What is the minimum content of an ICT business con...
    Faith:✓ Rel:✓ Guard:✓  The answer accurately extracts the requirements for the ICT 
  [4/17] [dora] What must the ICT risk management framework includ...
    Faith:✓ Rel:✓ Guard:✓  The answer accurately summarizes the requirements for the IC
  [5/17] [dora] What are the general principles for ICT third-part...
    Faith:✓ Rel:✓ Guard:✓  The answer accurately summarizes the general principles from
  [6/17] [dora] Is threat-led penetration testing mandatory for al...
    Faith:✓ Rel:✓ Guard:✓  The answer a

In [26]:
# Run this patch cell after Cell 8b to fix the citation flag on negative answers
for r in gen_results:
    if r['is_negative'] and re.search(
        r'INSUFFICIENT CONTEXT|outside the scope|does not contain',
        r['answer'], re.IGNORECASE
    ):
        r['has_citation'] = True   # refusal is correct output — not a citation failure

# Recompute cite_rate
cite_rate = sum(1 for r in gen_results if r['has_citation']) / len(gen_results)
print(f'✓ Citation rate after patch: {cite_rate:.0%}')

✓ Citation rate after patch: 100%


In [27]:
# Cell 9 — Generation metrics summary (per regulation + combined)

def gen_metrics(results, label):
    if not results:
        return {}
    valid = [r for r in results if r['faithful'] is not None]
    negs  = [r for r in results if r['is_negative']]
    faith_rate = sum(1 for r in valid if r['faithful'])  / len(valid)   if valid else 0
    rel_rate   = sum(1 for r in valid if r['relevant'])  / len(valid)   if valid else 0
    cite_rate  = sum(1 for r in results if r['has_citation']) / len(results)
    guard_rate = sum(1 for r in negs if r['hallucination_guard']) / len(negs) if negs else 1.0
    avg_lat    = round(sum(r['latency_s'] for r in results) / len(results), 2)
    print(f'  {label:<10}: n={len(results)}  Faith={faith_rate:.0%}  Rel={rel_rate:.0%}  Cite={cite_rate:.0%}  Guard={guard_rate:.0%}  Lat={avg_lat}s')
    return {'faith_rate': faith_rate, 'rel_rate': rel_rate, 'cite_rate': cite_rate,
             'guard_rate': guard_rate, 'avg_lat': avg_lat, 'n': len(results)}


print('GENERATION EVALUATION SUMMARY')
print('=' * 75)

dora_gen  = gen_metrics([r for r in gen_results if r.get('regulation') == 'dora'],   'DORA')
nis2_gen  = gen_metrics([r for r in gen_results if r.get('regulation') == 'nis2'],   'NIS2')
ps213_gen = gen_metrics([r for r in gen_results if r.get('regulation') == 'ps21_3'], 'PS21/3')

all_valid = [r for r in gen_results if r['faithful'] is not None]
all_negs  = [r for r in gen_results if r['is_negative']]
faith_rate  = sum(1 for r in all_valid if r['faithful'])  / len(all_valid) if all_valid else 0
rel_rate    = sum(1 for r in all_valid if r['relevant'])  / len(all_valid) if all_valid else 0
cite_rate   = sum(1 for r in gen_results if r['has_citation']) / len(gen_results)
oblig_rate  = sum(1 for r in gen_results if r['has_obligation']) / len(gen_results)
guard_rate  = sum(1 for r in all_negs if r['hallucination_guard']) / len(all_negs) if all_negs else 1.0
avg_lat     = round(sum(r['latency_s'] for r in gen_results) / len(gen_results), 2)

print()
print('  COMBINED:')
print(f'  Faithfulness     : {faith_rate:.0%}  (target ≥80%)')
print(f'  Relevancy        : {rel_rate:.0%}  (target ≥80%)')
print(f'  Citation rate    : {cite_rate:.0%}  (target ≥90%)')
print(f'  Obligation flag  : {oblig_rate:.0%}')
print(f'  Hallucin. guard  : {guard_rate:.0%}  (target 100%)')
print(f'  Avg latency      : {avg_lat}s')
print()
print('  Targets:')
print(f"  Faithfulness ≥80%   : {'✓' if faith_rate >= 0.8 else '✗'}")
print(f"  Relevancy    ≥80%   : {'✓' if rel_rate   >= 0.8 else '✗'}")
print(f"  Citation     ≥90%   : {'✓' if cite_rate  >= 0.9 else '✗'}")
print(f"  Guard        100%   : {'✓' if guard_rate == 1.0 else '✗'}")


GENERATION EVALUATION SUMMARY

  COMBINED:
  Faithfulness     : 100%  (target ≥80%)
  Relevancy        : 100%  (target ≥80%)
  Citation rate    : 100%  (target ≥90%)
  Obligation flag  : 90%
  Hallucin. guard  : 100%  (target 100%)
  Avg latency      : 9.49s

  Targets:
  Faithfulness ≥80%   : ✓
  Relevancy    ≥80%   : ✓
  Citation     ≥90%   : ✓
  Guard        100%   : ✓


In [28]:
# Cell 10 — Detailed answer review for failed cases
# Shows the actual answer for any question that failed a metric
# so you can diagnose whether it's a retrieval or generation issue

failed = [
    r for r in gen_results
    if not r["faithful"] or not r["relevant"]
    or not r["has_citation"]
    or (r["is_negative"] and not r["hallucination_guard"])
]

if not failed:
    print("✓ No failed cases — all questions passed all checks.")
else:
    print(f"{len(failed)} question(s) failed one or more checks:\n")
    for r in failed:
        print(f"{'═'*65}")
        print(f"Type    : {r['type']}")
        print(f"Question: {r['question']}")
        print(f"Metrics : Faith={r['faithful']}({r['faith_score']}) "
              f"Rel={r['relevant']}({r['rel_score']}) "
              f"Cite={r['has_citation']} "
              f"Guard={r['hallucination_guard']}")
        print(f"Answer  :\n{r['answer'][:500]}...")
        print()

✓ No failed cases — all questions passed all checks.


In [29]:
# Cell 11 — Save multi-regulation scorecard to disk
scorecard = {
    'created_at':  datetime.datetime.now().isoformat(),
    'pipeline':    'multi_regulation_v1',
    'llm':         GEMINI_MODEL,
    'corpus':      index_config.get('corpus_regulations', {}),
    'retrieval': {
        reg: {
            'n_questions':       ret.get('results') and len(ret['results']) or 0,
            'source_hit_rate':   round(ret.get('src_rate', 0), 3),
            'article_hit_rate':  round(ret.get('art_rate', 0), 3),
            'mrr':               round(ret.get('mrr', 0), 3),
            'avg_precision_at5': round(ret.get('avg_p5', 0), 3),
        }
        for reg, ret in retrieval_results_all.items() if ret
    },
    'generation': {
        'n_questions':        len(gen_results),
        'faithfulness_rate':  round(faith_rate, 3),
        'relevancy_rate':     round(rel_rate, 3),
        'citation_rate':      round(cite_rate, 3),
        'obligation_rate':    round(oblig_rate, 3),
        'hallucination_guard':round(guard_rate, 3),
        'avg_latency_s':      avg_lat,
        'by_regulation': {
            'dora':   dora_gen,
            'nis2':   nis2_gen,
            'ps21_3': ps213_gen,
        }
    },
}
scorecard_path = EVAL_DIR / f"scorecard_multiregulation_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}.json"
scorecard_path.write_text(json.dumps(scorecard, indent=2))
print(f'✓ Scorecard saved: {scorecard_path}')


✓ Scorecard saved: dora_rag/eval/scorecard_multiregulation_20260420_1819.json


In [30]:
# Cell 12 — Improvement recommendations
# Automatically generated based on scorecard results

print("IMPROVEMENT RECOMMENDATIONS")
print("=" * 60)
recs = []

if src_rate < 0.9:
    recs.append(("HIGH",   "Retrieval",   "Source hit rate below 90% — review chunking strategy for low-scoring query types"))
if art_rate < 0.7:
    recs.append(("MEDIUM", "Retrieval",   "Article hit rate below 70% — increase top_k from 8 to 10 or add re-ranking"))
if mrr < 0.5:
    recs.append(("HIGH",   "Retrieval",   "MRR below 0.5 — correct results buried in ranking, add a cross-encoder re-ranker"))
if faith_rate < 0.8:
    recs.append(("HIGH",   "Generation",  "Faithfulness below 80% — tighten prompt to prevent outside-knowledge usage"))
if rel_rate < 0.8:
    recs.append(("HIGH",   "Generation",  "Relevancy below 80% — review query routing rules or increase retrieval top_k"))
if cite_rate < 0.9:
    recs.append(("MEDIUM", "Generation",  "Citation rate below 90% — strengthen citation instruction in system prompt"))
if guard_rate < 1.0:
    recs.append(("HIGH",   "Hallucination","Hallucination guard not 100% — negative questions leaking answers, tighten prompt rule 5"))
if avg_lat > 10:
    recs.append(("LOW",    "Latency",      f"Avg latency {avg_lat}s — consider reducing context window or switching to a faster model"))

# Cross-level performance
cl_rows = by_type_results.get("cross_level", [])
if cl_rows and sum(1 for r in cl_rows if r["src_hit"]) / len(cl_rows) < 0.7:
    recs.append(("MEDIUM", "Retrieval",   "Cross-level queries underperforming — consider multi-retriever approach for L1+L2 queries"))

if not recs:
    print("  ✓ All metrics above target — pipeline is performing well at baseline.")
    print("  Consider moving to advanced improvements (re-ranking, HyDE, query expansion).")
else:
    print(f"  {'Priority':<8} {'Area':<15} Recommendation")
    print("  " + "─" * 72)
    for priority, area, rec in sorted(recs, key=lambda x: {"HIGH": 0, "MEDIUM": 1, "LOW": 2}[x[0]]):
        print(f"  {priority:<8} {area:<15} {rec}")

print(f"\n  Scorecard saved to: {scorecard_path}")
print(f"  Use this as your baseline before making any pipeline changes.")

IMPROVEMENT RECOMMENDATIONS
  ✓ All metrics above target — pipeline is performing well at baseline.
  Consider moving to advanced improvements (re-ranking, HyDE, query expansion).

  Scorecard saved to: dora_rag/eval/scorecard_multiregulation_20260420_1819.json
  Use this as your baseline before making any pipeline changes.


In [24]:
# Cell 13 — Phase 6b summary
print('\n' + '=' * 65)
print('PHASE 6b COMPLETE — MULTI-REGULATION EVALUATION FRAMEWORK')
print('=' * 65)
print(f"""
Evaluation question set : {len(EVAL_QUESTIONS)} questions
  DORA   : {len(DORA_QUESTIONS)} questions (30 original)
  NIS2   : {len(NIS2_QUESTIONS)} questions
  PS21/3 : {len(PS213_QUESTIONS)} questions
  Cross  : {len(CROSS_QUESTIONS)} comparison questions

RETRIEVAL SCORECARD:
  DORA   Source={dora_ret.get('src_rate',0):.0%}  Article={dora_ret.get('art_rate',0):.0%}  MRR={dora_ret.get('mrr',0):.3f}
  NIS2   Source={nis2_ret.get('src_rate',0):.0%}  Article={nis2_ret.get('art_rate',0):.0%}  MRR={nis2_ret.get('mrr',0):.3f}
  PS21/3 Source={ps213_ret.get('src_rate',0):.0%}  MRR={ps213_ret.get('mrr',0):.3f}

GENERATION SCORECARD (combined):
  Faithfulness  : {faith_rate:.0%}  (target ≥80%)
  Relevancy     : {rel_rate:.0%}  (target ≥80%)
  Citation rate : {cite_rate:.0%}  (target ≥90%)
  Halluc. guard : {guard_rate:.0%}  (target 100%)
  Avg latency   : {avg_lat}s

Judge            : {GEMINI_MODEL} (direct prompt — no LlamaIndex evaluators)
Scorecard saved  : {scorecard_path}

Next step → 08b_web_interface_multiregulation.ipynb
""")



PHASE 6b COMPLETE — MULTI-REGULATION EVALUATION FRAMEWORK

Evaluation question set : 70 questions
  DORA   : 30 questions (30 original)
  NIS2   : 15 questions
  PS21/3 : 15 questions
  Cross  : 10 comparison questions

RETRIEVAL SCORECARD:
  DORA   Source=93%  Article=89%  MRR=0.590
  NIS2   Source=92%  Article=83%  MRR=0.861
  PS21/3 Source=100%  MRR=0.874

GENERATION SCORECARD (combined):
  Faithfulness  : 100%  (target ≥80%)
  Relevancy     : 100%  (target ≥80%)
  Citation rate : 90%  (target ≥90%)
  Halluc. guard : 100%  (target 100%)
  Avg latency   : 9.49s

Judge            : models/gemini-flash-latest (direct prompt — no LlamaIndex evaluators)
Scorecard saved  : dora_rag/eval/scorecard_multiregulation_20260420_1817.json

Next step → 08b_web_interface_multiregulation.ipynb



In [25]:
# Inspect which questions failed faithfulness and why
failed_faith = [r for r in gen_results if r.get('faithful') == False]
print(f'Failed faithfulness: {len(failed_faith)}/{len(gen_results)}')
for r in failed_faith:
    print(f"\n  Q: {r['question'][:60]}")
    print(f"  Regulation: {r.get('regulation','?')}")
    print(f"  Reason: {r.get('reason','no reason given')}")
    print(f"  Cite: {r['has_citation']}  Guard: {r['hallucination_guard']}")

Failed faithfulness: 0/10
